### Web Scraping and Sentiment Analysis  
 
**Task:** scrape product reviews from Amazon, analyse customer sentiment, and train a machine-learning model to classify reviews as positive or negative.  
 
Write the Python code in phases or steps so I can run each cell on my Jupyter Notebook.
  
1. Web Scraping with BeautifulSoup to extract reviews.
2. Text Preprocessing & Cleaning (tokenisation, stopword removal).
3. Sentiment Analysis with VADER & TextBlob.
4. Machine Learning Sentiment Classification (using Naïve Bayes).
5. Visualizing Sentiment Trends using WordCloud & Bar Charts

In [ ]:
# Step 1: Web Scraping with BeautifulSoup to Extract Reviews

import requests
from bs4 import BeautifulSoup
import time

# Define the list of URLs to scrape (example: IMDb movie review pages; adapt for Amazon if classes match)
urls = [
    'https://www.imdb.com/title/tt0111161/reviews',  # Example: Shawshank Redemption
    # Add more URLs as needed, e.g., 'https://www.imdb.com/title/tt0068646/reviews'
]

# Initialize lists to store reviews and ratings
reviews_list = []
ratings_list = []

# Provided scraping code
for url in urls:
    print(f"Scraping: {url}")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'} 
    
    try:
        request = requests.get(url, headers=headers)
        page = BeautifulSoup(request.content, 'html.parser')

        # Find all review containers
        review_containers = page.find_all('div', class_='ipc-list-card__content')

        print(f"Found {len(review_containers)} reviews.")

    
        for container in review_containers:
            # Extract review text
            review_element = container.find('div', class_='ipc-html-content-inner-div')
            review_text = review_element.get_text(strip=True) if review_element else "No Review"
    
            # Extract rating
            rating_element = container.find('span', class_='ipc-rating-star--rating')
            rating = rating_element.get_text(strip=True) if rating_element else "NA"
    
            # Append to lists
            reviews_list.append(review_text)
            ratings_list.append(rating)
    
            # print(f"Review: {review_text[:100]}...")  # Print preview
            # print(f"Rating: {rating}\n")

    except Exception as e:
        print(f"Error scraping {url}: {e}")

    time.sleep(2)

# Print a summary
print(f"Total reviews scraped: {len(reviews_list)}")

Scraping: https://www.imdb.com/title/tt0111161/reviews
Found 25 reviews.
Total reviews scraped: 25


In [4]:
# Step 2: Text Preprocessing & Cleaning (Tokenization, Stopword Removal)

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

nltk.download('punkt_tab') 
nltk.download('stopwords')

# Define stopwords
stop_words = set(stopwords.words('english'))

# Function to preprocess a single review
def preprocess_review(text):
    if not text or text.strip() == "":
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and very short tokens
    filtered_tokens = [word for word in tokens 
                       if word not in stop_words and len(word) > 1]
    
    # Join back into string
    return ' '.join(filtered_tokens)

# Apply preprocessing to all reviews
cleaned_reviews = [preprocess_review(review) for review in reviews_list]

# Print summary
print(f"✅ Preprocessing completed!")
print(f"Total cleaned reviews: {len(cleaned_reviews)}")
print("\nSample cleaned review:")
print(cleaned_reviews[0] if cleaned_reviews else "No reviews found")

[nltk_data] Downloading package punkt_tab to /home/dev-
[nltk_data]     algo/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /home/dev-
[nltk_data]     algo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


✅ Preprocessing completed!
Total cleaned reviews: 25

Sample cleaned review:
wonder film high rating quite literally breathtaking say hasnt said much story acting premise movie makes feel sometimes watch film cant remember days later film loves youve seen dont forgetthe ultimate story friendship hope life overcoming adversityi understand many class best film time isnt mine get havent seen havent seen time need watch amazing 1010


In [5]:
# Step 3: Sentiment Analysis with VADER & TextBlob

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Initialize VADER
vader_analyzer = SentimentIntensityAnalyzer()

# Lists to store sentiments
vader_sentiments = []
textblob_sentiments = []

# Analyze each cleaned review
for review in cleaned_reviews:
    # VADER sentiment
    vader_score = vader_analyzer.polarity_scores(review)['compound']
    vader_label = 'positive' if vader_score > 0.05 else 'negative' if vader_score < -0.05 else 'neutral'
    vader_sentiments.append(vader_label)
    
    # TextBlob sentiment
    blob = TextBlob(review)
    blob_polarity = blob.sentiment.polarity
    blob_label = 'positive' if blob_polarity > 0 else 'negative' if blob_polarity < 0 else 'neutral'
    textblob_sentiments.append(blob_label)

# Print samples
print("Sample VADER sentiment:", vader_sentiments[0] if vader_sentiments else "No reviews")
print("Sample TextBlob sentiment:", textblob_sentiments[0] if textblob_sentiments else "No reviews")

ModuleNotFoundError: No module named 'vaderSentiment'

In [ ]:
# Step 4: Machine Learning Sentiment Classification (using Naïve Bayes)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Convert ratings to binary labels (assuming 1-10 scale; positive if >=7)
labels = []
for rating in ratings_list:
    if rating == 'NA':
        labels.append('positive')  # Default to positive or skip as needed
    else:
        try:
            rating_val = float(rating.split('/')[0])  # e.g., '9/10' -> 9.0
            labels.append('positive' if rating_val >= 7 else 'negative')
        except:
            labels.append('positive')  # Fallback

# Vectorize the cleaned reviews
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(cleaned_reviews)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

# Train Naïve Bayes
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train, y_train)

# Predict and evaluate
y_pred = nb_classifier.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Example: Predict on a new review
new_review = preprocess_review("This product is amazing!")
new_vec = vectorizer.transform([new_review])
print("Predicted sentiment for new review:", nb_classifier.predict(new_vec)[0])

In [ ]:
# Step 5: Visualizing Sentiment Trends using WordCloud & Bar Charts

from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter

# Combine positive and negative reviews for WordClouds
positive_reviews = ' '.join([cleaned_reviews[i] for i in range(len(vader_sentiments)) if vader_sentiments[i] == 'positive'])
negative_reviews = ' '.join([cleaned_reviews[i] for i in range(len(vader_sentiments)) if vader_sentiments[i] == 'negative'])

# Generate WordClouds
if positive_reviews:
    wc_pos = WordCloud(width=800, height=400, background_color='white').generate(positive_reviews)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc_pos, interpolation='bilinear')
    plt.title('Positive Reviews WordCloud')
    plt.axis('off')
    plt.show()

if negative_reviews:
    wc_neg = WordCloud(width=800, height=400, background_color='white').generate(negative_reviews)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc_neg, interpolation='bilinear')
    plt.title('Negative Reviews WordCloud')
    plt.axis('off')
    plt.show()

# Bar chart for sentiment distribution (using VADER)
sentiment_counts = Counter(vader_sentiments)
plt.figure(figsize=(8, 4))
plt.bar(sentiment_counts.keys(), sentiment_counts.values(), color=['green', 'red', 'blue'])
plt.title('Sentiment Distribution (VADER)')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()